任务一：生成关系分类数据集 任务一的目标是判断关系类型，标签为四分类：

0 表示“无关”； 1 表示“平行”； 2 表示“滞后”； 3 表示“超前”。

In [44]:
import random
import numpy as np
import pandas as pd
from torch import nn
import torch
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
set_seed(2025)

In [49]:
import numpy as np
import pandas as pd

def generate_time_series(T, sigma):
    """
    生成长度为T的时间序列，使用随机游走模型。
    :param T: 时间序列长度
    :param sigma: 随机噪声的标准差
    :return: 时间序列 (numpy array)
    """
    time_series = np.zeros(T)
    for t in range(1, T):
        time_series[t] = time_series[t - 1] + np.random.normal(0, sigma)
    return time_series

def generate_related_series(time_series_A, delay, sigma_B):
    """
    根据A的时间序列生成B的时间序列，加入时间延迟关系。
    :param time_series_A: 参照物A的时间序列
    :param delay: 时间延迟 (正值表示滞后，负值表示超前，0表示平行)
    :param sigma_B: B的随机噪声标准差
    :return: 时间序列B (numpy array)
    """
    T = len(time_series_A)
    time_series_B = np.zeros(T)
    for t in range(T):
        if 0 <= t - delay < T:
            time_series_B[t] = time_series_A[t - delay] + np.random.normal(0, sigma_B)
        else:
            time_series_B[t] = np.random.normal(0, sigma_B)
    return time_series_B

def generate_unrelated_series(T, sigma_B):
    """
    生成与A完全无关的B时间序列，仅由随机噪声组成。
    :param T: 时间序列长度
    :param sigma_B: 随机噪声的标准差
    :return: 时间序列B (numpy array)
    """
    return np.random.normal(0, sigma_B, T)

def generate_task1_dataset(N, T, sigma_A, sigma_B):
    """
    生成任务一的数据集，标签为四分类：无关(0)、平行(1)、滞后(2)、超前(3)。
    :param N: 总样本数量
    :param T: 每个时间序列的长度
    :param sigma_A: A的随机噪声标准差
    :param sigma_B: B的随机噪声标准差
    :return: 包含时间序列和标签的pandas DataFrame
    """
    categories = {
        "无关": {"label": 0, "delay": None},
        "平行": {"label": 1, "delay": [0]},
        "滞后": {"label": 2, "delay": [5, 10, 15]},
        "超前": {"label": 3, "delay": [-5, -10, -15]},
    }
    category_weights = {cat: 0.25 for cat in categories.keys()}  # 平衡分布
    data = []

    for _ in range(N):
        # 生成A的时间序列
        time_series_A = generate_time_series(T, sigma_A)

        # 随机选择关系类型
        category = np.random.choice(list(categories.keys()), p=list(category_weights.values()))
        label = categories[category]["label"]
        delay = None if category == "无关" else np.random.choice(categories[category]["delay"])

        # 根据关系类型生成B的时间序列
        if category == "无关":
            time_series_B = generate_unrelated_series(T, sigma_B)
        else:
            time_series_B = generate_related_series(time_series_A, delay, sigma_B)

        # 保存数据
        data.append({
            "time_series_A": time_series_A.tolist(),
            "time_series_B": time_series_B.tolist(),
            "relation_label": label,
        })

    return pd.DataFrame(data)

# 参数设置
N = 20000  # 样本总数
T = 30    # 时间序列长度
sigma_A = 5  # A的随机波动标准差
sigma_B = 5  # B的随机波动标准差

# 生成任务一数据集
task1_dataset = generate_task1_dataset(N, T, sigma_A, sigma_B)

# 保存为CSV
output_path_task1 = r"D:\mystudy\2024新版本\Thirdpaperdata\task1_dataset.csv"
task1_dataset.to_csv(output_path_task1, index=False)
print(f"任务一数据集已保存至 {output_path_task1}")


任务一数据集已保存至 D:\mystudy\2024新版本\Thirdpaperdata\task1_dataset.csv


任务二：生成时间延迟分类数据集 任务二的目标是对时间延迟进行具体分类预测，标签为八分类：

0 表示“无关”； 1 表示“平行”； 2 表示“滞后（5）”； 3 表示“滞后（10）”； 4 表示“滞后（20）”； 5 表示“超前（-5）”； 6 表示“超前（-10）”； 7 表示“超前（-20）”。

In [50]:
def generate_task2_dataset(N, T, sigma_A, sigma_B):
    """
    生成任务二的数据集，标签为八分类：
    无关(0)、平行(1)、滞后5(2)、滞后10(3)、滞后20(4)、超前-5(5)、超前-10(6)、超前-20(7)。
    :param N: 总样本数量
    :param T: 每个时间序列的长度
    :param sigma_A: A的随机噪声标准差
    :param sigma_B: B的随机噪声标准差
    :return: 包含时间序列和标签的pandas DataFrame
    """
    categories = {
        "无关": {"label": 0, "delay": None},
        "平行": {"label": 1, "delay": [0]},
        "滞后": {"label": {5: 2, 10: 3, 15: 4}, "delay": [5, 10, 15]},
        "超前": {"label": {-5: 5, -10: 6, -15: 7}, "delay": [-5, -10, -15]},
    }
    category_weights = {cat: 0.25 for cat in categories.keys()}  # 平衡分布
    data = []

    for _ in range(N):
        # 生成A的时间序列
        time_series_A = generate_time_series(T, sigma_A)

        # 随机选择关系类型
        category = np.random.choice(list(categories.keys()), p=list(category_weights.values()))
        if category == "无关":
            label = 0
            delay = None
            time_series_B = generate_unrelated_series(T, sigma_B)
        elif category == "平行":
            label = 1
            delay = 0
            time_series_B = generate_related_series(time_series_A, delay, sigma_B)
        else:  # 滞后或超前
            delay = np.random.choice(categories[category]["delay"])
            label = categories[category]["label"][delay]
            time_series_B = generate_related_series(time_series_A, delay, sigma_B)

        # 保存数据
        data.append({
            "time_series_A": time_series_A.tolist(),
            "time_series_B": time_series_B.tolist(),
            "delay_label": label,
        })

    return pd.DataFrame(data)

# 生成任务二数据集
task2_dataset = generate_task2_dataset(N, T, sigma_A, sigma_B)

# 保存为CSV
output_path_task2 = r"D:\mystudy\2024新版本\Thirdpaperdata\task2_dataset.csv"
task2_dataset.to_csv(output_path_task2, index=False)
print(f"任务二数据集已保存至 {output_path_task2}")


任务二数据集已保存至 D:\mystudy\2024新版本\Thirdpaperdata\task2_dataset.csv


任务1 线性回归不含检测器

In [79]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score, f1_score

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
set_seed(2025)
# 数据导入
task1_data_path = r"D:\mystudy\2024新版本\Thirdpaperdata\task1_dataset.csv"
data = pd.read_csv(task1_data_path)

# 特征提取函数
def extract_features_with_lags(time_series_A, time_series_B, delays=[-15, -10, -5, 0, 5, 10, 15]):
    """
    提取特征：计算不同超前滞后项下的差异特征。
    :param time_series_A: A的时间序列（列表格式）
    :param time_series_B: B的时间序列（列表格式）
    :param delays: 延迟的取值范围
    :return: 特征向量 (numpy array)
    """
    series_A = np.array(eval(time_series_A))
    series_B = np.array(eval(time_series_B))
    features = []

    for delay in delays:
        # 滞后 A 或超前 A 的时间序列
        if delay > 0:
            shifted_A = np.roll(series_A, delay)
            shifted_A[:delay] = 0  # 前面的空缺位置填充0
        elif delay < 0:
            shifted_A = np.roll(series_A, delay)
            shifted_A[delay:] = 0  # 后面的空缺位置填充0
        else:  # delay == 0
            shifted_A = series_A

        # 计算差异特征
        diff = series_B - shifted_A
        features.append(diff)

    # 将所有延迟特征拼接在一起
    return np.concatenate(features)

# 提取特征和标签
X = np.array([extract_features_with_lags(row["time_series_A"], row["time_series_B"]) for _, row in data.iterrows()])
y = data["relation_label"].values  # 标签列

# 数据集划分
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, stratify=y)

# 模型训练
print("开始训练线性回归模型...")
linear_model = LogisticRegression(max_iter=1000)
linear_model.fit(X_train, y_train)
print("训练完成！")

# 模型预测
y_pred = linear_model.predict(X_test)

# 评估结果
print("\n评估结果：")
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(f"Macro F1 Score: {f1_score(y_test, y_pred, average='macro'):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=["Unrelated", "Parallel", "Lagging", "Leading"]))

# 特征数量检查
print(f"特征数量: {X.shape[1]}")

开始训练线性回归模型...
训练完成！

评估结果：
Accuracy: 0.2720
Macro F1 Score: 0.2718

Classification Report:
              precision    recall  f1-score   support

   Unrelated       0.28      0.29      0.29      1520
    Parallel       0.29      0.26      0.27      1493
     Lagging       0.26      0.28      0.27      1500
     Leading       0.26      0.25      0.26      1487

    accuracy                           0.27      6000
   macro avg       0.27      0.27      0.27      6000
weighted avg       0.27      0.27      0.27      6000

特征数量: 210


In [80]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score, f1_score
import torch


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
set_seed(2025)
# 数据导入
task1_data_path = r"D:\mystudy\2024新版本\Thirdpaperdata\task1_dataset.csv"
data = pd.read_csv(task1_data_path)

# 超前滞后检测器
def compute_lag_with_encoding(time_series_A, time_series_B, max_lag=20, n_hid=16):
    """
    检测超前滞后关系并进行时间编码。
    :param time_series_A: A 的时间序列（字符串格式）
    :param time_series_B: B 的时间序列（字符串格式）
    :param max_lag: 最大滞后步
    :param n_hid: 编码器的隐层维度
    :return: 时间编码器生成的特征向量 (numpy array)
    """
    series_A = np.array(eval(time_series_A))
    series_B = np.array(eval(time_series_B))

    # 标准化时间序列
    series_A = (series_A - np.mean(series_A)) / np.std(series_A)
    series_B = (series_B - np.mean(series_B)) / np.std(series_B)

    # 计算互相关
    cross_correlation = np.correlate(series_B, series_A, mode='full')
    mid = len(cross_correlation) // 2
    relevant_range = cross_correlation[mid - max_lag: mid + max_lag + 1]

    # 找到最优滞后步
    optimal_lag = np.argmax(np.abs(relevant_range)) - max_lag

    # 时间编码器
    delta_t = torch.tensor([optimal_lag], dtype=torch.float32)
    dim_t = torch.arange(0, n_hid // 2, dtype=torch.float32)
    div_term = torch.exp((2 * dim_t) / n_hid * (-np.log(10000.0)))
    sin_term = torch.sin(delta_t.unsqueeze(-1) * div_term)
    cos_term = torch.cos(delta_t.unsqueeze(-1) * div_term)
    time_encoding = torch.cat([sin_term, cos_term], dim=-1).numpy()

    return time_encoding.flatten()

# 提取特征函数（新增超前滞后特征）
def extract_features_with_lag_detector_and_encoding(time_series_A, time_series_B, delays=[-15, -10, -5, 0, 5, 10, 15], n_hid=16):
    """
    提取特征：包括原始的延迟特征和时间编码器生成的超前滞后特征。
    :param time_series_A: A 的时间序列（字符串格式）
    :param time_series_B: B 的时间序列（字符串格式）
    :param delays: 延迟的取值范围
    :param n_hid: 时间编码器的隐层维度
    :return: 特征向量 (numpy array)
    """
    # 计算超前滞后特征
    time_encoding_features = compute_lag_with_encoding(time_series_A, time_series_B, max_lag=20, n_hid=n_hid)

    # 基于现有逻辑提取差异特征
    series_A = np.array(eval(time_series_A))
    series_B = np.array(eval(time_series_B))
    diff_features = []

    for delay in delays:
        if delay > 0:
            shifted_A = np.roll(series_A, delay)
            shifted_A[:delay] = 0
        elif delay < 0:
            shifted_A = np.roll(series_A, delay)
            shifted_A[delay:] = 0
        else:
            shifted_A = series_A

        diff = series_B - shifted_A
        diff_features.append(diff)

    # 合并检测器特征与差异特征
    return np.concatenate([time_encoding_features, np.concatenate(diff_features)])

# 提取增强特征和标签
X = np.array([extract_features_with_lag_detector_and_encoding(row["time_series_A"], row["time_series_B"]) for _, row in data.iterrows()])
y = data["relation_label"].values  # 标签列

# 数据集划分
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, stratify=y)

# 模型训练
print("开始训练含超前滞后检测器的线性回归模型...")
linear_model_with_lag_detector = LogisticRegression(max_iter=1000)
linear_model_with_lag_detector.fit(X_train, y_train)
print("训练完成！")

# 模型预测
y_pred = linear_model_with_lag_detector.predict(X_test)

# 评估结果
print("\n评估结果（含检测器）：")
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(f"Macro F1 Score: {f1_score(y_test, y_pred, average='macro'):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=["Unrelated", "Parallel", "Lagging", "Leading"]))

# 特征数量检查
print(f"特征数量: {X.shape[1]}")


开始训练含超前滞后检测器的线性回归模型...
训练完成！

评估结果（含检测器）：
Accuracy: 0.5873
Macro F1 Score: 0.5631

Classification Report:
              precision    recall  f1-score   support

   Unrelated       0.48      0.24      0.32      1520
    Parallel       0.77      0.99      0.87      1493
     Lagging       0.46      0.56      0.50      1500
     Leading       0.57      0.57      0.57      1487

    accuracy                           0.59      6000
   macro avg       0.57      0.59      0.56      6000
weighted avg       0.57      0.59      0.56      6000

特征数量: 226


D:\anconda\Lib\site-packages\sklearn\linear_model\_logistic.py:458: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


任务1 线性回归不含检测器

In [81]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score, f1_score

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
set_seed(2025)

# 数据导入
task2_data_path = r"D:\mystudy\2024新版本\Thirdpaperdata\task2_dataset.csv"
data = pd.read_csv(task2_data_path)

# 特征提取函数
def extract_features_with_lags(time_series_A, time_series_B, delays=[-15, -10, -5, 0, 5, 10, 15]):
    """
    提取特征：计算不同超前滞后项下的差异特征。
    :param time_series_A: A的时间序列（列表格式）
    :param time_series_B: B的时间序列（列表格式）
    :param delays: 延迟的取值范围
    :return: 特征向量 (numpy array)
    """
    series_A = np.array(eval(time_series_A))
    series_B = np.array(eval(time_series_B))
    features = []

    for delay in delays:
        # 滞后 A 或超前 A 的时间序列
        if delay > 0:
            shifted_A = np.roll(series_A, delay)
            shifted_A[:delay] = 0  # 前面的空缺位置填充0
        elif delay < 0:
            shifted_A = np.roll(series_A, delay)
            shifted_A[delay:] = 0  # 后面的空缺位置填充0
        else:  # delay == 0
            shifted_A = series_A

        # 计算差异特征
        diff = series_B - shifted_A
        features.append(diff)

    # 将所有延迟特征拼接在一起
    return np.concatenate(features)

# 提取特征和标签
X = np.array([extract_features_with_lags(row["time_series_A"], row["time_series_B"]) for _, row in data.iterrows()])
y = data["delay_label"].values  # 标签列 (0: 无关, 1: 平行, 2-7: 滞后/超前)

# 数据集划分
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, stratify=y)

# 模型训练
print("开始训练线性回归模型（任务2，八分类）...")
linear_model = LogisticRegression(max_iter=1000, multi_class='multinomial')
linear_model.fit(X_train, y_train)
print("训练完成！")

# 模型预测
y_pred = linear_model.predict(X_test)

# 评估结果
print("\n评估结果：")
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(f"Macro F1 Score: {f1_score(y_test, y_pred, average='macro'):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=[
    "Unrelated", "Parallel", "Lagging-5", "Lagging-10", "Lagging-20",
    "Leading-5", "Leading-10", "Leading-20"
]))

# 特征数量检查
print(f"特征数量: {X.shape[1]}")


开始训练线性回归模型（任务2，八分类）...
训练完成！

评估结果：
Accuracy: 0.2462
Macro F1 Score: 0.0823

Classification Report:
              precision    recall  f1-score   support

   Unrelated       0.25      0.54      0.34      1500
    Parallel       0.24      0.46      0.32      1459
   Lagging-5       0.00      0.00      0.00       519
  Lagging-10       0.00      0.00      0.00       509
  Lagging-20       0.00      0.00      0.00       517
   Leading-5       0.00      0.00      0.00       478
  Leading-10       0.00      0.00      0.00       525
  Leading-20       0.00      0.00      0.00       493

    accuracy                           0.25      6000
   macro avg       0.06      0.12      0.08      6000
weighted avg       0.12      0.25      0.16      6000

特征数量: 210


D:\anconda\Lib\site-packages\sklearn\metrics\_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
D:\anconda\Lib\site-packages\sklearn\metrics\_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
D:\anconda\Lib\site-packages\sklearn\metrics\_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


In [82]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score, f1_score
import torch
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
set_seed(2025)
# 数据导入
task2_data_path = r"D:\mystudy\2024新版本\Thirdpaperdata\task2_dataset.csv"
data = pd.read_csv(task2_data_path)

# 超前滞后检测器
def compute_lag_with_encoding(time_series_A, time_series_B, max_lag=20, n_hid=16):
    """
    检测超前滞后关系并进行时间编码。
    :param time_series_A: A 的时间序列（字符串格式）
    :param time_series_B: B 的时间序列（字符串格式）
    :param max_lag: 最大滞后步
    :param n_hid: 编码器的隐层维度
    :return: 时间编码器生成的特征向量 (numpy array)
    """
    series_A = np.array(eval(time_series_A))
    series_B = np.array(eval(time_series_B))

    # 标准化时间序列
    series_A = (series_A - np.mean(series_A)) / np.std(series_A)
    series_B = (series_B - np.mean(series_B)) / np.std(series_B)

    # 计算互相关
    cross_correlation = np.correlate(series_B, series_A, mode='full')
    mid = len(cross_correlation) // 2
    relevant_range = cross_correlation[mid - max_lag: mid + max_lag + 1]

    # 找到最优滞后步
    optimal_lag = np.argmax(np.abs(relevant_range)) - max_lag

    # 时间编码器
    delta_t = torch.tensor([optimal_lag], dtype=torch.float32)
    dim_t = torch.arange(0, n_hid // 2, dtype=torch.float32)
    div_term = torch.exp((2 * dim_t) / n_hid * (-np.log(10000.0)))
    sin_term = torch.sin(delta_t.unsqueeze(-1) * div_term)
    cos_term = torch.cos(delta_t.unsqueeze(-1) * div_term)
    time_encoding = torch.cat([sin_term, cos_term], dim=-1).numpy()

    return time_encoding.flatten()

# 提取超前滞后特征与原始数据
def prepare_features(data, n_hid=16):
    """
    组合超前滞后效应编码和原始时间序列。
    :param data: 输入数据集
    :param n_hid: 时间编码器的隐层维度
    :return: 特征矩阵 (numpy array)
    """
    features = []
    for _, row in data.iterrows():
        # 编码超前滞后效应
        lag_encoding = compute_lag_with_encoding(row["time_series_A"], row["time_series_B"], max_lag=20, n_hid=n_hid)
        
        # 原始时间序列（展平后加入特征向量）
        series_A = np.array(eval(row["time_series_A"])).flatten()
        series_B = np.array(eval(row["time_series_B"])).flatten()
        
        # 合并特征
        combined_features = np.concatenate([lag_encoding, series_A, series_B])
        features.append(combined_features)
    
    return np.array(features)

# 准备数据
X = prepare_features(data, n_hid=16)
y = data["delay_label"].values  # 标签列 (0: 无关, 1: 平行, 2-7: 滞后/超前)

# 数据集划分
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3,  stratify=y)

# 模型训练
print("开始训练线性回归模型（任务2，含超前滞后效应和原始数据）...")
linear_model = LogisticRegression(max_iter=1000, multi_class='multinomial')
linear_model.fit(X_train, y_train)
print("训练完成！")

# 模型预测
y_pred = linear_model.predict(X_test)

# 评估结果
print("\n评估结果：")
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(f"Macro F1 Score: {f1_score(y_test, y_pred, average='macro'):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=[
    "Unrelated", "Parallel", "Lagging-5", "Lagging-10", "Lagging-20",
    "Leading-5", "Leading-10", "Leading-20"
]))

# 特征数量检查
print(f"特征数量: {X.shape[1]}")


开始训练线性回归模型（任务2，含超前滞后效应和原始数据）...
训练完成！

评估结果：
Accuracy: 0.6135
Macro F1 Score: 0.5267

Classification Report:
              precision    recall  f1-score   support

   Unrelated       0.48      0.58      0.53      1500
    Parallel       0.78      1.00      0.88      1459
   Lagging-5       0.47      0.64      0.54       519
  Lagging-10       0.55      0.25      0.34       509
  Lagging-20       0.71      0.13      0.22       517
   Leading-5       0.65      0.84      0.73       478
  Leading-10       0.62      0.52      0.56       525
  Leading-20       0.61      0.30      0.41       493

    accuracy                           0.61      6000
   macro avg       0.61      0.53      0.53      6000
weighted avg       0.62      0.61      0.58      6000

特征数量: 76


D:\anconda\Lib\site-packages\sklearn\linear_model\_logistic.py:458: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


任务一 DNN 不加检测器

In [95]:
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score, f1_score
from torch.utils.data import DataLoader, TensorDataset

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
set_seed(2025)

# 数据导入
task1_data_path = r"D:\mystudy\2024新版本\Thirdpaperdata\task1_dataset.csv"
data = pd.read_csv(task1_data_path)

# 提取特征和标签
# 直接使用原始时间序列数据
X = np.array([np.concatenate([np.array(eval(row["time_series_A"])), np.array(eval(row["time_series_B"]))]) for _, row in data.iterrows()])
y = data["relation_label"].values  # 标签列 (0: 无关, 1: 平行, 2: 滞后, 3: 超前)

# 数据集划分
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, stratify=y)

# 转换为 PyTorch 张量
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.long)
y_test_tensor = torch.tensor(y_test, dtype=torch.long)

# 数据加载器
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

#定义三层全连接神经网络
class FCNN(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim):
        super(FCNN, self).__init__()
        self.model = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, output_dim)
        )
    
    def forward(self, x):
        return self.model(x)
    

# 初始化模型
input_dim = X_train.shape[1]  # 输入特征维度
hidden_dim = 128  # 隐藏层维度
output_dim = 4  # 分类数（无关、平行、滞后、超前）

model = FCNN(input_dim, hidden_dim, output_dim)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

# 损失函数和优化器
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# 训练模型
def train_model(model, train_loader, criterion, optimizer, device, epochs=20):
    model.train()
    for epoch in range(epochs):
        total_loss = 0
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)

            # 前向传播
            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)

            # 反向传播和优化
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_loss += loss.item()
        print(f"Epoch {epoch+1}/{epochs}, Loss: {total_loss:.4f}")

# 测试模型
def evaluate_model(model, test_loader, device):
    model.eval()
    all_preds = []
    all_labels = []
    with torch.no_grad():
        for X_batch, y_batch in test_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            outputs = model(X_batch)
            _, preds = torch.max(outputs, 1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(y_batch.cpu().numpy())
    return np.array(all_labels), np.array(all_preds)

# 训练
print("开始训练三层全连接神经网络...")
train_model(model, train_loader, criterion, optimizer, device, epochs=20)
print("训练完成！")

# 测试
y_true, y_pred = evaluate_model(model, test_loader, device)

# 评估
print("\n评估结果：")
print(f"Accuracy: {accuracy_score(y_true, y_pred):.5f}")
print(f"Macro F1 Score: {f1_score(y_true, y_pred, average='macro'):.5f}")
print("\nClassification Report:")
print(classification_report(y_true, y_pred, target_names=["Unrelated", "Parallel", "Lagging", "Leading"]))

开始训练三层全连接神经网络...
Epoch 1/20, Loss: 80.1616
Epoch 2/20, Loss: 24.9234
Epoch 3/20, Loss: 18.1945
Epoch 4/20, Loss: 14.4589
Epoch 5/20, Loss: 11.2867
Epoch 6/20, Loss: 9.4007
Epoch 7/20, Loss: 7.5825
Epoch 8/20, Loss: 10.0922
Epoch 9/20, Loss: 8.3595
Epoch 10/20, Loss: 5.0445
Epoch 11/20, Loss: 3.6958
Epoch 12/20, Loss: 3.8163
Epoch 13/20, Loss: 1.7171
Epoch 14/20, Loss: 3.5361
Epoch 15/20, Loss: 2.8373
Epoch 16/20, Loss: 14.9842
Epoch 17/20, Loss: 5.2546
Epoch 18/20, Loss: 1.2100
Epoch 19/20, Loss: 1.7529
Epoch 20/20, Loss: 0.8633
训练完成！

评估结果：
Accuracy: 0.96567
Macro F1 Score: 0.96573

Classification Report:
              precision    recall  f1-score   support

   Unrelated       0.96      0.93      0.95      1520
    Parallel       0.99      0.99      0.99      1493
     Lagging       0.94      0.96      0.95      1500
     Leading       0.98      0.97      0.97      1487

    accuracy                           0.97      6000
   macro avg       0.97      0.97      0.97      6000
weight

In [96]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score, f1_score
from torch.utils.data import DataLoader, TensorDataset

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
set_seed(2025)
# 数据导入
task1_data_path = r"D:\mystudy\2024新版本\Thirdpaperdata\task1_dataset.csv"
data = pd.read_csv(task1_data_path)

# 超前滞后检测器
def compute_lag_with_encoding(time_series_A, time_series_B, max_lag=20, n_hid=16):
    """
    检测超前滞后关系并进行时间编码。
    :param time_series_A: A 的时间序列（字符串格式）
    :param time_series_B: B 的时间序列（字符串格式）
    :param max_lag: 最大滞后步
    :param n_hid: 编码器的隐层维度
    :return: 时间编码器生成的特征向量 (numpy array)
    """
    series_A = np.array(eval(time_series_A))
    series_B = np.array(eval(time_series_B))

    # 标准化时间序列
    series_A = (series_A - np.mean(series_A)) / np.std(series_A)
    series_B = (series_B - np.mean(series_B)) / np.std(series_B)

    # 计算互相关
    cross_correlation = np.correlate(series_B, series_A, mode='full')
    mid = len(cross_correlation) // 2
    relevant_range = cross_correlation[mid - max_lag: mid + max_lag + 1]

    # 找到最优滞后步
    optimal_lag = np.argmax(np.abs(relevant_range)) - max_lag

    # 时间编码器
    delta_t = torch.tensor([optimal_lag], dtype=torch.float32)
    dim_t = torch.arange(0, n_hid // 2, dtype=torch.float32)
    div_term = torch.exp((2 * dim_t) / n_hid * (-np.log(10000.0)))
    sin_term = torch.sin(delta_t.unsqueeze(-1) * div_term)
    cos_term = torch.cos(delta_t.unsqueeze(-1) * div_term)
    time_encoding = torch.cat([sin_term, cos_term], dim=-1).numpy()

    return time_encoding.flatten()

# 提取特征和标签
# 组合原始时间序列和编码的超前滞后效应
def prepare_features(data, n_hid=16):
    """
    组合原始时间序列数据和编码的超前滞后效应。
    """
    features = []
    for _, row in data.iterrows():
        # 原始时间序列数据
        series_A = np.array(eval(row["time_series_A"]))
        series_B = np.array(eval(row["time_series_B"]))
        original_features = np.concatenate([series_A, series_B])

        # 编码的超前滞后效应
        lag_encoding = compute_lag_with_encoding(row["time_series_A"], row["time_series_B"], n_hid=n_hid)

        # 合并特征
        combined_features = np.concatenate([original_features, lag_encoding])
        features.append(combined_features)
    return np.array(features)

# 准备数据
X = prepare_features(data, n_hid=16)
y = data["relation_label"].values  # 标签列 (0: 无关, 1: 平行, 2: 滞后, 3: 超前)

# 数据集划分
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, stratify=y)

# 转换为 PyTorch 张量
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.long)
y_test_tensor = torch.tensor(y_test, dtype=torch.long)

# 数据加载器
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

# 定义三层全连接神经网络
class FCNN(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim):
        super(FCNN, self).__init__()
        self.model = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, output_dim)
        )
    
    def forward(self, x):
        return self.model(x)
    
        
# 初始化模型
input_dim = X_train.shape[1]  # 输入特征维度（包括原始数据和编码特征）
hidden_dim = 128  # 隐藏层维度
output_dim = 4  # 分类数（无关、平行、滞后、超前）

model = FCNN(input_dim, hidden_dim, output_dim)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

# 损失函数和优化器
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# 训练模型
def train_model(model, train_loader, criterion, optimizer, device, epochs=20):
    model.train()
    for epoch in range(epochs):
        total_loss = 0
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)

            # 前向传播
            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)

            # 反向传播和优化
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_loss += loss.item()
        print(f"Epoch {epoch+1}/{epochs}, Loss: {total_loss:.4f}")

# 测试模型
def evaluate_model(model, test_loader, device):
    model.eval()
    all_preds = []
    all_labels = []
    with torch.no_grad():
        for X_batch, y_batch in test_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            outputs = model(X_batch)
            _, preds = torch.max(outputs, 1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(y_batch.cpu().numpy())
    return np.array(all_labels), np.array(all_preds)

# 训练
print("开始训练三层全连接神经网络...")
train_model(model, train_loader, criterion, optimizer, device, epochs=20)
print("训练完成！")

# 测试
y_true, y_pred = evaluate_model(model, test_loader, device)

# 评估
print("\n评估结果：")
print(f"Accuracy: {accuracy_score(y_true, y_pred):.5f}")
print(f"Macro F1 Score: {f1_score(y_true, y_pred, average='macro'):.5f}")
print("\nClassification Report:")
print(classification_report(y_true, y_pred, target_names=["Unrelated", "Parallel", "Lagging", "Leading"]))


开始训练三层全连接神经网络...
Epoch 1/20, Loss: 65.2426
Epoch 2/20, Loss: 21.5304
Epoch 3/20, Loss: 18.1640
Epoch 4/20, Loss: 11.2642
Epoch 5/20, Loss: 8.2898
Epoch 6/20, Loss: 8.2387
Epoch 7/20, Loss: 9.1103
Epoch 8/20, Loss: 4.3541
Epoch 9/20, Loss: 4.4533
Epoch 10/20, Loss: 6.6363
Epoch 11/20, Loss: 7.3566
Epoch 12/20, Loss: 3.2054
Epoch 13/20, Loss: 1.5751
Epoch 14/20, Loss: 2.3010
Epoch 15/20, Loss: 1.5861
Epoch 16/20, Loss: 14.4063
Epoch 17/20, Loss: 2.4452
Epoch 18/20, Loss: 1.9571
Epoch 19/20, Loss: 0.9970
Epoch 20/20, Loss: 1.0634
训练完成！

评估结果：
Accuracy: 0.97450
Macro F1 Score: 0.97457

Classification Report:
              precision    recall  f1-score   support

   Unrelated       0.95      0.97      0.96      1520
    Parallel       0.99      1.00      0.99      1493
     Lagging       0.98      0.96      0.97      1500
     Leading       0.98      0.98      0.98      1487

    accuracy                           0.97      6000
   macro avg       0.97      0.97      0.97      6000
weighted

任务二 DNN 不加检测器

In [97]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score, f1_score
from torch.utils.data import DataLoader, TensorDataset

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
set_seed(2025)

# 数据导入
task2_data_path = r"D:\mystudy\2024新版本\Thirdpaperdata\task2_dataset.csv"
data = pd.read_csv(task2_data_path)

# 提取特征和标签
# 直接使用原始时间序列数据
X = np.array([np.concatenate([np.array(eval(row["time_series_A"])), np.array(eval(row["time_series_B"]))]) for _, row in data.iterrows()])
y = data["delay_label"].values  # 标签列 (0: 无关, 1: 平行, 2-7: 滞后/超前)

# 数据集划分
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, stratify=y)

# 转换为 PyTorch 张量
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.long)
y_test_tensor = torch.tensor(y_test, dtype=torch.long)

# 数据加载器
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

# 定义三层全连接神经网络
class FCNN(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim):
        super(FCNN, self).__init__()
        self.model = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, output_dim)
        )
    
    def forward(self, x):
        return self.model(x)

# 初始化模型
input_dim = X_train.shape[1]  # 输入特征维度
hidden_dim = 128  # 隐藏层维度
output_dim = 8  # 分类数（无关、平行、滞后5/10/20、超前-5/-10/-20）

model = FCNN(input_dim, hidden_dim, output_dim)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

# 损失函数和优化器
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# 训练模型
def train_model(model, train_loader, criterion, optimizer, device, epochs=20):
    model.train()
    for epoch in range(epochs):
        total_loss = 0
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)

            # 前向传播
            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)

            # 反向传播和优化
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_loss += loss.item()
        print(f"Epoch {epoch+1}/{epochs}, Loss: {total_loss:.4f}")

# 测试模型
def evaluate_model(model, test_loader, device):
    model.eval()
    all_preds = []
    all_labels = []
    with torch.no_grad():
        for X_batch, y_batch in test_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            outputs = model(X_batch)
            _, preds = torch.max(outputs, 1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(y_batch.cpu().numpy())
    return np.array(all_labels), np.array(all_preds)

# 训练
print("开始训练三层全连接神经网络（八分类）...")
train_model(model, train_loader, criterion, optimizer, device, epochs=20)
print("训练完成！")

# 测试
y_true, y_pred = evaluate_model(model, test_loader, device)

# 评估
print("\n评估结果：")
print(f"Accuracy: {accuracy_score(y_true, y_pred):.4f}")
print(f"Macro F1 Score: {f1_score(y_true, y_pred, average='macro'):.4f}")
print("\nClassification Report:")
print(classification_report(y_true, y_pred, target_names=[
    "Unrelated", "Parallel", "Lagging-5", "Lagging-10", "Lagging-20",
    "Leading-5", "Leading-10", "Leading-20"
]))

开始训练三层全连接神经网络（八分类）...
Epoch 1/20, Loss: 153.4031
Epoch 2/20, Loss: 47.0902
Epoch 3/20, Loss: 29.0467
Epoch 4/20, Loss: 22.0778
Epoch 5/20, Loss: 16.5805
Epoch 6/20, Loss: 12.0681
Epoch 7/20, Loss: 15.1305
Epoch 8/20, Loss: 11.5709
Epoch 9/20, Loss: 8.2731
Epoch 10/20, Loss: 8.6544
Epoch 11/20, Loss: 4.4722
Epoch 12/20, Loss: 3.7961
Epoch 13/20, Loss: 11.3771
Epoch 14/20, Loss: 8.9753
Epoch 15/20, Loss: 4.3667
Epoch 16/20, Loss: 3.4836
Epoch 17/20, Loss: 9.8890
Epoch 18/20, Loss: 3.2163
Epoch 19/20, Loss: 0.6138
Epoch 20/20, Loss: 0.3160
训练完成！

评估结果：
Accuracy: 0.9698
Macro F1 Score: 0.9660

Classification Report:
              precision    recall  f1-score   support

   Unrelated       0.96      0.97      0.96      1500
    Parallel       0.99      1.00      0.99      1459
   Lagging-5       0.98      0.98      0.98       519
  Lagging-10       0.96      0.95      0.96       509
  Lagging-20       0.95      0.92      0.93       517
   Leading-5       0.98      0.97      0.98       478
 

In [98]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score, f1_score
from torch.utils.data import DataLoader, TensorDataset

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
set_seed(2025)

# 数据导入
task2_data_path = r"D:\mystudy\2024新版本\Thirdpaperdata\task2_dataset.csv"
data = pd.read_csv(task2_data_path)

# 超前滞后检测器
def compute_lag_with_encoding(time_series_A, time_series_B, max_lag=20, n_hid=16):
    """
    检测超前滞后关系并进行时间编码。
    :param time_series_A: A 的时间序列（字符串格式）
    :param time_series_B: B 的时间序列（字符串格式）
    :param max_lag: 最大滞后步
    :param n_hid: 编码器的隐层维度
    :return: 时间编码器生成的特征向量 (numpy array)
    """
    series_A = np.array(eval(time_series_A))
    series_B = np.array(eval(time_series_B))

    # 标准化时间序列
    series_A = (series_A - np.mean(series_A)) / np.std(series_A)
    series_B = (series_B - np.mean(series_B)) / np.std(series_B)

    # 计算互相关
    cross_correlation = np.correlate(series_B, series_A, mode='full')
    mid = len(cross_correlation) // 2
    relevant_range = cross_correlation[mid - max_lag: mid + max_lag + 1]

    # 找到最优滞后步
    optimal_lag = np.argmax(np.abs(relevant_range)) - max_lag

    # 时间编码器
    delta_t = torch.tensor([optimal_lag], dtype=torch.float32)
    dim_t = torch.arange(0, n_hid // 2, dtype=torch.float32)
    div_term = torch.exp((2 * dim_t) / n_hid * (-np.log(10000.0)))
    sin_term = torch.sin(delta_t.unsqueeze(-1) * div_term)
    cos_term = torch.cos(delta_t.unsqueeze(-1) * div_term)
    time_encoding = torch.cat([sin_term, cos_term], dim=-1).numpy()

    return time_encoding.flatten()

# 提取特征和标签
# 组合原始时间序列和编码的超前滞后效应
def prepare_features(data, n_hid=16):
    """
    组合原始时间序列数据和编码的超前滞后效应。
    """
    features = []
    for _, row in data.iterrows():
        # 原始时间序列数据
        series_A = np.array(eval(row["time_series_A"]))
        series_B = np.array(eval(row["time_series_B"]))
        original_features = np.concatenate([series_A, series_B])

        # 编码的超前滞后效应
        lag_encoding = compute_lag_with_encoding(row["time_series_A"], row["time_series_B"], n_hid=n_hid)

        # 合并特征
        combined_features = np.concatenate([original_features, lag_encoding])
        features.append(combined_features)
    return np.array(features)

# 准备数据
X = prepare_features(data, n_hid=16)
y = data["delay_label"].values  # 标签列 (0: 无关, 1: 平行, 2-7: 滞后/超前)

# 数据集划分
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, stratify=y)

# 转换为 PyTorch 张量
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.long)
y_test_tensor = torch.tensor(y_test, dtype=torch.long)

# 数据加载器
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

# 定义三层全连接神经网络
class FCNN(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim):
        super(FCNN, self).__init__()
        self.model = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, output_dim)
        )
    
    def forward(self, x):
        return self.model(x)

# 初始化模型
input_dim = X_train.shape[1]  # 输入特征维度（包括原始数据和编码特征）
hidden_dim = 128  # 隐藏层维度
output_dim = 8  # 分类数（无关、平行、滞后5/10/20、超前-5/-10/-20）

model = FCNN(input_dim, hidden_dim, output_dim)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

# 损失函数和优化器
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# 训练模型
def train_model(model, train_loader, criterion, optimizer, device, epochs=30):
    model.train()
    for epoch in range(epochs):
        total_loss = 0
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)

            # 前向传播
            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)

            # 反向传播和优化
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_loss += loss.item()
        print(f"Epoch {epoch+1}/{epochs}, Loss: {total_loss:.4f}")

# 测试模型
def evaluate_model(model, test_loader, device):
    model.eval()
    all_preds = []
    all_labels = []
    with torch.no_grad():
        for X_batch, y_batch in test_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            outputs = model(X_batch)
            _, preds = torch.max(outputs, 1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(y_batch.cpu().numpy())
    return np.array(all_labels), np.array(all_preds)

# 训练
print("开始训练三层全连接神经网络（八分类）...")
train_model(model, train_loader, criterion, optimizer, device, epochs=20)
print("训练完成！")

# 测试
y_true, y_pred = evaluate_model(model, test_loader, device)

# 评估
print("\n评估结果：")
print(f"Accuracy: {accuracy_score(y_true, y_pred):.4f}")
print(f"Macro F1 Score: {f1_score(y_true, y_pred, average='macro'):.4f}")
print("\nClassification Report:")
print(classification_report(y_true, y_pred, target_names=[
    "Unrelated", "Parallel", "Lagging-5", "Lagging-10", "Lagging-20",
    "Leading-5", "Leading-10", "Leading-20"
]))


开始训练三层全连接神经网络（八分类）...
Epoch 1/20, Loss: 140.3862
Epoch 2/20, Loss: 39.6004
Epoch 3/20, Loss: 25.2323
Epoch 4/20, Loss: 17.9524
Epoch 5/20, Loss: 14.8844
Epoch 6/20, Loss: 12.8521
Epoch 7/20, Loss: 9.5398
Epoch 8/20, Loss: 7.4404
Epoch 9/20, Loss: 13.2297
Epoch 10/20, Loss: 6.9203
Epoch 11/20, Loss: 9.7723
Epoch 12/20, Loss: 5.1947
Epoch 13/20, Loss: 3.3778
Epoch 14/20, Loss: 3.2845
Epoch 15/20, Loss: 2.5938
Epoch 16/20, Loss: 1.0315
Epoch 17/20, Loss: 1.0821
Epoch 18/20, Loss: 0.6157
Epoch 19/20, Loss: 0.4020
Epoch 20/20, Loss: 0.1966
训练完成！

评估结果：
Accuracy: 0.9725
Macro F1 Score: 0.9697

Classification Report:
              precision    recall  f1-score   support

   Unrelated       0.95      0.97      0.96      1500
    Parallel       0.99      1.00      1.00      1459
   Lagging-5       0.98      0.99      0.98       519
  Lagging-10       0.96      0.95      0.96       509
  Lagging-20       0.96      0.90      0.93       517
   Leading-5       0.98      0.99      0.98       478
  L

任务一 GRU 不加检测器

In [67]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score, f1_score
from torch.utils.data import DataLoader, TensorDataset
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
set_seed(2025)
# 数据导入
task1_data_path = r"D:\mystudy\2024新版本\Thirdpaperdata\task1_dataset.csv"
data = pd.read_csv(task1_data_path)

# 提取特征和标签
# 直接使用原始时间序列数据
X_A = np.array([np.array(eval(row["time_series_A"])) for _, row in data.iterrows()])
X_B = np.array([np.array(eval(row["time_series_B"])) for _, row in data.iterrows()])
y = data["relation_label"].values  # 标签列 (0: 无关, 1: 平行, 2: 滞后, 3: 超前)

# 合并时间序列 A 和 B，形状为 (样本数, 时间步长, 2)
X = np.concatenate((X_A[..., np.newaxis], X_B[..., np.newaxis]), axis=2)

# 数据集划分
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, stratify=y)

# 转换为 PyTorch 张量
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.long)
y_test_tensor = torch.tensor(y_test, dtype=torch.long)

# 数据加载器
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

# 定义 GRU 模型
class GRUNet(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim, num_layers=1, dropout=0.2):
        super(GRUNet, self).__init__()
        self.gru = nn.GRU(input_size=input_dim, hidden_size=hidden_dim, num_layers=num_layers, batch_first=True, dropout=dropout)
        self.fc = nn.Linear(hidden_dim, output_dim)
    
    def forward(self, x):
        _, h_n = self.gru(x)  # h_n: (num_layers, batch, hidden_dim)
        h_n = h_n[-1]  # 取最后一层的隐藏状态
        out = self.fc(h_n)
        return out

# 初始化模型
input_dim = 2  # 每个时间步的特征维度（A 和 B 两个时间序列）
hidden_dim = 128  # 隐藏层维度
output_dim = 4  # 分类数（无关、平行、滞后、超前）
num_layers = 2  # GRU 层数

model = GRUNet(input_dim, hidden_dim, output_dim, num_layers=num_layers)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

# 损失函数和优化器
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# 训练模型
def train_model(model, train_loader, criterion, optimizer, device, epochs=20):
    model.train()
    for epoch in range(epochs):
        total_loss = 0
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)

            # 前向传播
            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)

            # 反向传播和优化
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_loss += loss.item()
        print(f"Epoch {epoch+1}/{epochs}, Loss: {total_loss:.4f}")

# 测试模型
def evaluate_model(model, test_loader, device):
    model.eval()
    all_preds = []
    all_labels = []
    with torch.no_grad():
        for X_batch, y_batch in test_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            outputs = model(X_batch)
            _, preds = torch.max(outputs, 1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(y_batch.cpu().numpy())
    return np.array(all_labels), np.array(all_preds)

# 训练
print("开始训练 GRU 模型（四分类）...")
train_model(model, train_loader, criterion, optimizer, device, epochs=30)
print("训练完成！")

# 测试
y_true, y_pred = evaluate_model(model, test_loader, device)

# 评估
print("\n评估结果：")
print(f"Accuracy: {accuracy_score(y_true, y_pred):.4f}")
print(f"Macro F1 Score: {f1_score(y_true, y_pred, average='macro'):.4f}")
print("\nClassification Report:")
print(classification_report(y_true, y_pred, target_names=[
    "Unrelated", "Parallel", "Lagging", "Leading"
])) 

开始训练 GRU 模型（四分类）...
Epoch 1/30, Loss: 68.3559
Epoch 2/30, Loss: 19.3615
Epoch 3/30, Loss: 16.3280
Epoch 4/30, Loss: 12.8504
Epoch 5/30, Loss: 12.3885
Epoch 6/30, Loss: 12.3219
Epoch 7/30, Loss: 10.8938
Epoch 8/30, Loss: 10.1721
Epoch 9/30, Loss: 9.7367
Epoch 10/30, Loss: 8.3608
Epoch 11/30, Loss: 8.4524
Epoch 12/30, Loss: 7.9836
Epoch 13/30, Loss: 6.9995
Epoch 14/30, Loss: 6.2137
Epoch 15/30, Loss: 5.8264
Epoch 16/30, Loss: 5.5920
Epoch 17/30, Loss: 6.4500
Epoch 18/30, Loss: 5.5351
Epoch 19/30, Loss: 4.1768
Epoch 20/30, Loss: 4.3806
Epoch 21/30, Loss: 2.6477
Epoch 22/30, Loss: 3.5068
Epoch 23/30, Loss: 4.3403
Epoch 24/30, Loss: 2.2092
Epoch 25/30, Loss: 2.9079
Epoch 26/30, Loss: 1.5051
Epoch 27/30, Loss: 1.4717
Epoch 28/30, Loss: 0.7617
Epoch 29/30, Loss: 2.5139
Epoch 30/30, Loss: 1.8674
训练完成！

评估结果：
Accuracy: 0.9757
Macro F1 Score: 0.9758

Classification Report:
              precision    recall  f1-score   support

   Unrelated       0.96      0.97      0.96      1520
    Parallel   

In [68]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score, f1_score
from torch.utils.data import DataLoader, TensorDataset

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
set_seed(2025)

# 数据导入
task1_data_path = r"D:\mystudy\2024新版本\Thirdpaperdata\task1_dataset.csv"
data = pd.read_csv(task1_data_path)

# 超前滞后检测器
def compute_lag_with_encoding(time_series_A, time_series_B, max_lag=20, n_hid=16):
    """
    检测超前滞后关系并进行时间编码。
    :param time_series_A: A 的时间序列（numpy array）
    :param time_series_B: B 的时间序列（numpy array）
    :param max_lag: 最大滞后步
    :param n_hid: 编码器的隐层维度
    :return: 时间编码器生成的特征向量 (numpy array)
    """
    series_A = (time_series_A - np.mean(time_series_A)) / np.std(time_series_A)
    series_B = (time_series_B - np.mean(time_series_B)) / np.std(time_series_B)

    # 计算互相关
    cross_correlation = np.correlate(series_B, series_A, mode='full')
    mid = len(cross_correlation) // 2
    relevant_range = cross_correlation[mid - max_lag: mid + max_lag + 1]

    # 找到最优滞后步
    optimal_lag = np.argmax(np.abs(relevant_range)) - max_lag

    # 时间编码器
    delta_t = torch.tensor([optimal_lag], dtype=torch.float32)
    dim_t = torch.arange(0, n_hid // 2, dtype=torch.float32)
    div_term = torch.exp((2 * dim_t) / n_hid * (-np.log(10000.0)))
    sin_term = torch.sin(delta_t.unsqueeze(-1) * div_term)
    cos_term = torch.cos(delta_t.unsqueeze(-1) * div_term)
    time_encoding = torch.cat([sin_term, cos_term], dim=-1).numpy()

    return time_encoding.flatten()

# 提取特征和标签
X_A = np.array([np.array(eval(row["time_series_A"])) for _, row in data.iterrows()])
X_B = np.array([np.array(eval(row["time_series_B"])) for _, row in data.iterrows()])
y = data["relation_label"].values  # 标签列 (0: 无关, 1: 平行, 2: 滞后, 3: 超前)

# 提取超前滞后特征并调整形状
lag_features = np.array([compute_lag_with_encoding(X_A[i], X_B[i]) for i in range(len(X_A))])  # (样本数, 编码特征维度)
time_steps = X_A.shape[1]  # 时间步长

# 将 lag_features 调整为与时间步长匹配的形状
lag_features_expanded = np.repeat(lag_features[:, np.newaxis, :], time_steps, axis=1)  # (样本数, 时间步长, 编码特征维度)

# 将 X_A 和 X_B 扩展为与 lag_features_expanded 匹配的形状
X_A_expanded = X_A[..., np.newaxis]  # (样本数, 时间步长, 1)
X_B_expanded = X_B[..., np.newaxis]  # (样本数, 时间步长, 1)

# 拼接 X_A, X_B 和 lag_features
X_combined = np.concatenate((X_A_expanded, X_B_expanded, lag_features_expanded), axis=2)  # (样本数, 时间步长, 特征维度)


# 数据集划分
X_train, X_test, y_train, y_test = train_test_split(X_combined, y, test_size=0.3,  stratify=y)

# 转换为 PyTorch 张量
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.long)
y_test_tensor = torch.tensor(y_test, dtype=torch.long)

# 数据加载器
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

# 定义 GRU 模型
class GRUNet(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim, num_layers=1, dropout=0.2):
        super(GRUNet, self).__init__()
        self.gru = nn.GRU(input_size=input_dim, hidden_size=hidden_dim, num_layers=num_layers, batch_first=True, dropout=dropout)
        self.fc = nn.Linear(hidden_dim, output_dim)
    
    def forward(self, x):
        _, h_n = self.gru(x)  # h_n: (num_layers, batch, hidden_dim)
        h_n = h_n[-1]  # 取最后一层的隐藏状态
        out = self.fc(h_n)
        return out

# 初始化模型
input_dim = X_combined.shape[2]  # 每个时间步的特征维度（A 和 B + 编码特征）
hidden_dim = 128  # 隐藏层维度
output_dim = 4  # 分类数（无关、平行、滞后、超前）
num_layers = 2  # GRU 层数

model = GRUNet(input_dim, hidden_dim, output_dim, num_layers=num_layers)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

# 损失函数和优化器
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# 训练模型
def train_model(model, train_loader, criterion, optimizer, device, epochs=30):
    model.train()
    for epoch in range(epochs):
        total_loss = 0
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)

            # 前向传播
            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)

            # 反向传播和优化
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_loss += loss.item()
        print(f"Epoch {epoch+1}/{epochs}, Loss: {total_loss:.4f}")

# 测试模型
def evaluate_model(model, test_loader, device):
    model.eval()
    all_preds = []
    all_labels = []
    with torch.no_grad():
        for X_batch, y_batch in test_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            outputs = model(X_batch)
            _, preds = torch.max(outputs, 1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(y_batch.cpu().numpy())
    return np.array(all_labels), np.array(all_preds)

# 训练
print("开始训练 GRU 模型（四分类）...")
train_model(model, train_loader, criterion, optimizer, device, epochs=30)
print("训练完成！")

# 测试
y_true, y_pred = evaluate_model(model, test_loader, device)

# 评估
print("\n评估结果：")
print(f"Accuracy: {accuracy_score(y_true, y_pred):.4f}")
print(f"Macro F1 Score: {f1_score(y_true, y_pred, average='macro'):.4f}")
print("\nClassification Report:")
print(classification_report(y_true, y_pred, target_names=[
    "Unrelated", "Parallel", "Lagging", "Leading"
]))


开始训练 GRU 模型（四分类）...
Epoch 1/30, Loss: 60.6594
Epoch 2/30, Loss: 16.9597
Epoch 3/30, Loss: 13.8300
Epoch 4/30, Loss: 12.0031
Epoch 5/30, Loss: 11.2159
Epoch 6/30, Loss: 9.8521
Epoch 7/30, Loss: 9.1580
Epoch 8/30, Loss: 8.8529
Epoch 9/30, Loss: 8.0495
Epoch 10/30, Loss: 7.8509
Epoch 11/30, Loss: 7.1147
Epoch 12/30, Loss: 6.4735
Epoch 13/30, Loss: 6.3567
Epoch 14/30, Loss: 5.9996
Epoch 15/30, Loss: 5.5737
Epoch 16/30, Loss: 4.3591
Epoch 17/30, Loss: 4.5521
Epoch 18/30, Loss: 3.0929
Epoch 19/30, Loss: 3.5434
Epoch 20/30, Loss: 3.4885
Epoch 21/30, Loss: 2.8951
Epoch 22/30, Loss: 2.6317
Epoch 23/30, Loss: 2.0793
Epoch 24/30, Loss: 1.7216
Epoch 25/30, Loss: 1.2091
Epoch 26/30, Loss: 4.1291
Epoch 27/30, Loss: 3.1392
Epoch 28/30, Loss: 1.7916
Epoch 29/30, Loss: 0.7446
Epoch 30/30, Loss: 0.8083
训练完成！

评估结果：
Accuracy: 0.9790
Macro F1 Score: 0.9791

Classification Report:
              precision    recall  f1-score   support

   Unrelated       0.96      0.97      0.96      1520
    Parallel      

任务二 GRU 不加检测器

In [72]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score, f1_score
from torch.utils.data import DataLoader, TensorDataset

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
set_seed(2025)

# 数据导入
task2_data_path = r"D:\mystudy\2024新版本\Thirdpaperdata\task2_dataset.csv"
data = pd.read_csv(task2_data_path)

# 直接使用原始时间序列数据
X_A = np.array([np.array(eval(row["time_series_A"])) for _, row in data.iterrows()])
X_B = np.array([np.array(eval(row["time_series_B"])) for _, row in data.iterrows()])
y = data["delay_label"].values  # 标签列 (0: 无关, 1: 平行, 2-7: 滞后/超前)

# 将两个时间序列堆叠在一起，形状为 (样本数, 1, 时间步长 * 2)
X = np.stack((X_A, X_B), axis=2).transpose((0, 2, 1))  # 形状为 (样本数, 时间步长 * 2, 1)

# 数据集划分
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3,  stratify=y)

# 转换为 PyTorch 张量
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.long)
y_test_tensor = torch.tensor(y_test, dtype=torch.long)

# 数据加载器
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

# 定义 GRU 模型
class GRUNet(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim, num_layers=1, dropout=0.2):
        super(GRUNet, self).__init__()
        self.gru = nn.GRU(input_dim, hidden_dim, num_layers, batch_first=True, dropout=dropout)
        self.fc = nn.Linear(hidden_dim, output_dim)
    
    def forward(self, x):
        _, h_n = self.gru(x)  # h_n: (num_layers, batch, hidden_dim)
        h_n = h_n[-1]  # 取最后一层的隐藏状态
        out = self.fc(h_n)
        return out

# 初始化模型
input_dim = X_train.shape[2]  # 每个时间步的特征维度（A 和 B 两个时间序列）
hidden_dim = 128  # 隐藏层维度
output_dim = 8  # 分类数（无关、平行、滞后5/10/20、超前-5/-10/-20）
num_layers = 2  # GRU 层数

model = GRUNet(input_dim, hidden_dim, output_dim, num_layers=num_layers)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

# 损失函数和优化器
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# 训练模型
def train_model(model, train_loader, criterion, optimizer, device, epochs=20):
    model.train()
    for epoch in range(epochs):
        total_loss = 0
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)

            # 前向传播
            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)

            # 反向传播和优化
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_loss += loss.item()
        print(f"Epoch {epoch+1}/{epochs}, Loss: {total_loss:.4f}")

# 测试模型
def evaluate_model(model, test_loader, device):
    model.eval()
    all_preds = []
    all_labels = []
    with torch.no_grad():
        for X_batch, y_batch in test_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            outputs = model(X_batch)
            _, preds = torch.max(outputs, 1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(y_batch.cpu().numpy())
    return np.array(all_labels), np.array(all_preds)

# 训练
print("开始训练 GRU 模型（八分类）...")
train_model(model, train_loader, criterion, optimizer, device, epochs=30)
print("训练完成！")

# 测试
y_true, y_pred = evaluate_model(model, test_loader, device)

# 评估
print("\n评估结果：")
print(f"Accuracy: {accuracy_score(y_true, y_pred):.4f}")
print(f"Macro F1 Score: {f1_score(y_true, y_pred, average='macro'):.4f}")
print("\nClassification Report:")
print(classification_report(y_true, y_pred, target_names=[
    "Unrelated", "Parallel", "Lagging-5", "Lagging-10", "Lagging-20",
    "Leading-5", "Leading-10", "Leading-20"
]))

开始训练 GRU 模型（八分类）...
Epoch 1/30, Loss: 217.9233
Epoch 2/30, Loss: 90.9703
Epoch 3/30, Loss: 64.5427
Epoch 4/30, Loss: 50.0697
Epoch 5/30, Loss: 40.9306
Epoch 6/30, Loss: 32.8444
Epoch 7/30, Loss: 28.5674
Epoch 8/30, Loss: 25.3526
Epoch 9/30, Loss: 23.8857
Epoch 10/30, Loss: 20.2145
Epoch 11/30, Loss: 18.4527
Epoch 12/30, Loss: 17.6897
Epoch 13/30, Loss: 18.0052
Epoch 14/30, Loss: 15.4638
Epoch 15/30, Loss: 13.9569
Epoch 16/30, Loss: 14.4589
Epoch 17/30, Loss: 13.7475
Epoch 18/30, Loss: 14.5064
Epoch 19/30, Loss: 15.7638
Epoch 20/30, Loss: 13.1319
Epoch 21/30, Loss: 10.6023
Epoch 22/30, Loss: 10.8051
Epoch 23/30, Loss: 10.7557
Epoch 24/30, Loss: 12.6746
Epoch 25/30, Loss: 9.9851
Epoch 26/30, Loss: 10.6129
Epoch 27/30, Loss: 10.2929
Epoch 28/30, Loss: 11.5497
Epoch 29/30, Loss: 8.7849
Epoch 30/30, Loss: 11.0094
训练完成！

评估结果：
Accuracy: 0.9340
Macro F1 Score: 0.9219

Classification Report:
              precision    recall  f1-score   support

   Unrelated       0.93      0.96      0.95     

In [71]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score, f1_score
from torch.utils.data import DataLoader, TensorDataset
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
set_seed(2025)


# 数据导入
task2_data_path = r"D:\mystudy\2024新版本\Thirdpaperdata\task2_dataset.csv"
data = pd.read_csv(task2_data_path)

# 超前滞后检测器
def compute_lag_with_encoding(time_series_A, time_series_B, max_lag=20, n_hid=16):
    """
    检测超前滞后关系并进行时间编码。
    :param time_series_A: A 的时间序列（numpy array）
    :param time_series_B: B 的时间序列（numpy array）
    :param max_lag: 最大滞后步
    :param n_hid: 编码器的隐层维度
    :return: 时间编码器生成的特征向量 (numpy array)
    """
    series_A = (time_series_A - np.mean(time_series_A)) / np.std(time_series_A)
    series_B = (time_series_B - np.mean(time_series_B)) / np.std(time_series_B)

    # 计算互相关
    cross_correlation = np.correlate(series_B, series_A, mode='full')
    mid = len(cross_correlation) // 2
    relevant_range = cross_correlation[mid - max_lag: mid + max_lag + 1]

    # 找到最优滞后步
    optimal_lag = np.argmax(np.abs(relevant_range)) - max_lag

    # 时间编码器
    delta_t = torch.tensor([optimal_lag], dtype=torch.float32)
    dim_t = torch.arange(0, n_hid // 2, dtype=torch.float32)
    div_term = torch.exp((2 * dim_t) / n_hid * (-np.log(10000.0)))
    sin_term = torch.sin(delta_t.unsqueeze(-1) * div_term)
    cos_term = torch.cos(delta_t.unsqueeze(-1) * div_term)
    time_encoding = torch.cat([sin_term, cos_term], dim=-1).numpy()

    return time_encoding.flatten()

# 提取原始时间序列和超前滞后特征
X_A = np.array([np.array(eval(row["time_series_A"])) for _, row in data.iterrows()])
X_B = np.array([np.array(eval(row["time_series_B"])) for _, row in data.iterrows()])
y = data["delay_label"].values  # 标签列 (0: 无关, 1: 平行, 2-7: 滞后/超前)

# 提取超前滞后特征并拼接到时间序列
lag_features = np.array([compute_lag_with_encoding(X_A[i], X_B[i]) for i in range(len(X_A))])  # (样本数, 编码特征维度)
time_steps = X_A.shape[1]  # 时间步长

# 将 lag_features 扩展为与时间步长匹配的形状
lag_features_expanded = np.repeat(lag_features[:, np.newaxis, :], time_steps, axis=1)  # (样本数, 时间步长, 编码特征维度)

# 将 X_A 和 X_B 扩展为与 lag_features_expanded 匹配的形状
X_A_expanded = X_A[..., np.newaxis]  # (样本数, 时间步长, 1)
X_B_expanded = X_B[..., np.newaxis]  # (样本数, 时间步长, 1)

# 拼接 X_A, X_B 和 lag_features
X_combined = np.concatenate((X_A_expanded, X_B_expanded, lag_features_expanded), axis=2)  # (样本数, 时间步长, 特征维度)

# 数据集划分
X_train, X_test, y_train, y_test = train_test_split(X_combined, y, test_size=0.3, stratify=y)

# 转换为 PyTorch 张量
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.long)
y_test_tensor = torch.tensor(y_test, dtype=torch.long)

# 数据加载器
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

# 定义 GRU 模型
class GRUNet(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim, num_layers=1, dropout=0.2):
        super(GRUNet, self).__init__()
        self.gru = nn.GRU(input_size=input_dim, hidden_size=hidden_dim, num_layers=num_layers, batch_first=True, dropout=dropout)
        self.fc = nn.Linear(hidden_dim, output_dim)
    
    def forward(self, x):
        _, h_n = self.gru(x)  # h_n: (num_layers, batch, hidden_dim)
        h_n = h_n[-1]  # 取最后一层的隐藏状态
        out = self.fc(h_n)
        return out

# 初始化模型
input_dim = X_combined.shape[2]  # 每个时间步的特征维度（A 和 B + 编码特征）
hidden_dim = 128  # 隐藏层维度
output_dim = 8  # 分类数（无关、平行、滞后5/10/20、超前-5/-10/-20）
num_layers = 2  # GRU 层数

model = GRUNet(input_dim, hidden_dim, output_dim, num_layers=num_layers)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

# 损失函数和优化器
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# 训练模型
def train_model(model, train_loader, criterion, optimizer, device, epochs=20):
    model.train()
    for epoch in range(epochs):
        total_loss = 0
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)

            # 前向传播
            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)

            # 反向传播和优化
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_loss += loss.item()
        print(f"Epoch {epoch+1}/{epochs}, Loss: {total_loss:.4f}")

# 测试模型
def evaluate_model(model, test_loader, device):
    model.eval()
    all_preds = []
    all_labels = []
    with torch.no_grad():
        for X_batch, y_batch in test_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            outputs = model(X_batch)
            _, preds = torch.max(outputs, 1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(y_batch.cpu().numpy())
    return np.array(all_labels), np.array(all_preds)

# 训练
print("开始训练 GRU 模型（八分类）...")
train_model(model, train_loader, criterion, optimizer, device, epochs=30)
print("训练完成！")

# 测试
y_true, y_pred = evaluate_model(model, test_loader, device)

# 评估
print("\n评估结果：")
print(f"Accuracy: {accuracy_score(y_true, y_pred):.4f}")
print(f"Macro F1 Score: {f1_score(y_true, y_pred, average='macro'):.4f}")
print("\nClassification Report:")
print(classification_report(y_true, y_pred, target_names=[
    "Unrelated", "Parallel", "Lagging-5", "Lagging-10", "Lagging-20",
    "Leading-5", "Leading-10", "Leading-20"
]))


开始训练 GRU 模型（八分类）...
Epoch 1/30, Loss: 137.8596
Epoch 2/30, Loss: 46.3300
Epoch 3/30, Loss: 35.2722
Epoch 4/30, Loss: 29.8193
Epoch 5/30, Loss: 25.2515
Epoch 6/30, Loss: 22.4857
Epoch 7/30, Loss: 19.9270
Epoch 8/30, Loss: 19.7340
Epoch 9/30, Loss: 17.2962
Epoch 10/30, Loss: 16.7633
Epoch 11/30, Loss: 14.2827
Epoch 12/30, Loss: 13.6204
Epoch 13/30, Loss: 11.5287
Epoch 14/30, Loss: 11.4404
Epoch 15/30, Loss: 10.1320
Epoch 16/30, Loss: 11.2626
Epoch 17/30, Loss: 10.4380
Epoch 18/30, Loss: 8.5704
Epoch 19/30, Loss: 6.8506
Epoch 20/30, Loss: 7.6867
Epoch 21/30, Loss: 5.8524
Epoch 22/30, Loss: 6.5920
Epoch 23/30, Loss: 5.4922
Epoch 24/30, Loss: 3.5684
Epoch 25/30, Loss: 6.0094
Epoch 26/30, Loss: 4.9680
Epoch 27/30, Loss: 4.2150
Epoch 28/30, Loss: 3.3657
Epoch 29/30, Loss: 3.8281
Epoch 30/30, Loss: 4.7387
训练完成！

评估结果：
Accuracy: 0.9623
Macro F1 Score: 0.9528

Classification Report:
              precision    recall  f1-score   support

   Unrelated       0.96      0.98      0.97      1500
    P

GRU+CNN 任务1

In [73]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score, f1_score
from torch.utils.data import DataLoader, TensorDataset

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
set_seed(2025)

# 数据导入
task1_data_path = r"D:\mystudy\2024新版本\Thirdpaperdata\task1_dataset.csv"
data = pd.read_csv(task1_data_path)

# 直接使用原始时间序列数据
X_A = np.array([np.array(eval(row["time_series_A"])) for _, row in data.iterrows()])
X_B = np.array([np.array(eval(row["time_series_B"])) for _, row in data.iterrows()])
y = data["relation_label"].values  # 标签列 (0: 无关, 1: 平行, 2: 滞后, 3: 超前)

# 将两个时间序列堆叠在一起，形状为 (样本数, 时间步长, 2)
X = np.concatenate((X_A[..., np.newaxis], X_B[..., np.newaxis]), axis=2)

# 数据集划分
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3,  stratify=y)

# 转换为 PyTorch 张量
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.long)
y_test_tensor = torch.tensor(y_test, dtype=torch.long)

# 数据加载器
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

# 定义 CNN + GRU 模型
class CNN_GRU_Net(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim, num_layers=1, kernel_size=3, dropout=0.2):
        super(CNN_GRU_Net, self).__init__()
        self.conv1d = nn.Conv1d(in_channels=2, out_channels=64, kernel_size=kernel_size, padding=1)
        self.relu = nn.ReLU()
        self.gru = nn.GRU(input_size=64, hidden_size=hidden_dim, num_layers=num_layers, batch_first=True, dropout=dropout)
        self.fc = nn.Linear(hidden_dim, output_dim)
    
    def forward(self, x):
        # 输入形状: (batch, seq_len, input_dim)
        x = x.permute(0, 2, 1)  # 转换为 (batch, input_dim, seq_len) 以适配 Conv1d
        x = self.conv1d(x)      # 经过 1D 卷积
        x = self.relu(x)
        x = x.permute(0, 2, 1)  # 转换回 (batch, seq_len, conv_features) 以适配 GRU
        _, h_n = self.gru(x)    # h_n: (num_layers, batch, hidden_dim)
        h_n = h_n[-1]           # 取最后一层的隐藏状态
        out = self.fc(h_n)      # 全连接层分类
        return out

# 初始化模型
input_dim = 2  # 每个时间步的特征维度（A 和 B 两个时间序列）
hidden_dim = 128  # 隐藏层维度
output_dim = 4  # 分类数（无关、平行、滞后、超前）
num_layers = 1  # GRU 层数
kernel_size = 3  # 卷积核大小

model = CNN_GRU_Net(input_dim, hidden_dim, output_dim, num_layers=num_layers, kernel_size=kernel_size)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

# 损失函数和优化器
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# 训练模型
def train_model(model, train_loader, criterion, optimizer, device, epochs=20):
    model.train()
    for epoch in range(epochs):
        total_loss = 0
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)

            # 前向传播
            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)

            # 反向传播和优化
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_loss += loss.item()
        print(f"Epoch {epoch+1}/{epochs}, Loss: {total_loss:.4f}")

# 测试模型
def evaluate_model(model, test_loader, device):
    model.eval()
    all_preds = []
    all_labels = []
    with torch.no_grad():
        for X_batch, y_batch in test_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            outputs = model(X_batch)
            _, preds = torch.max(outputs, 1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(y_batch.cpu().numpy())
    return np.array(all_labels), np.array(all_preds)

# 训练
print("开始训练 CNN + GRU 模型（四分类）...")
train_model(model, train_loader, criterion, optimizer, device, epochs=30)
print("训练完成！")

# 测试
y_true, y_pred = evaluate_model(model, test_loader, device)

# 评估
print("\n评估结果：")
print(f"Accuracy: {accuracy_score(y_true, y_pred):.4f}")
print(f"Macro F1 Score: {f1_score(y_true, y_pred, average='macro'):.4f}")
print("\nClassification Report:")
print(classification_report(y_true, y_pred, target_names=[
    "Unrelated", "Parallel", "Lagging", "Leading"
])) 

C:\Users\10340\AppData\Roaming\Python\Python311\site-packages\torch\nn\modules\rnn.py:71: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.2 and num_layers=1
  warnings.warn("dropout option adds dropout after all but last "


开始训练 CNN + GRU 模型（四分类）...
Epoch 1/30, Loss: 68.9856
Epoch 2/30, Loss: 20.2912
Epoch 3/30, Loss: 14.9333
Epoch 4/30, Loss: 13.6602
Epoch 5/30, Loss: 11.5896
Epoch 6/30, Loss: 9.7106
Epoch 7/30, Loss: 10.4870
Epoch 8/30, Loss: 7.6957
Epoch 9/30, Loss: 6.9277
Epoch 10/30, Loss: 6.1563
Epoch 11/30, Loss: 5.5819
Epoch 12/30, Loss: 4.0017
Epoch 13/30, Loss: 4.2980
Epoch 14/30, Loss: 3.8754
Epoch 15/30, Loss: 3.7220
Epoch 16/30, Loss: 3.3830
Epoch 17/30, Loss: 2.6567
Epoch 18/30, Loss: 2.0840
Epoch 19/30, Loss: 1.1824
Epoch 20/30, Loss: 2.4000
Epoch 21/30, Loss: 2.2418
Epoch 22/30, Loss: 2.2587
Epoch 23/30, Loss: 0.8133
Epoch 24/30, Loss: 1.6596
Epoch 25/30, Loss: 2.5322
Epoch 26/30, Loss: 1.7390
Epoch 27/30, Loss: 0.4967
Epoch 28/30, Loss: 0.7195
Epoch 29/30, Loss: 0.7146
Epoch 30/30, Loss: 2.3808
训练完成！

评估结果：
Accuracy: 0.9742
Macro F1 Score: 0.9743

Classification Report:
              precision    recall  f1-score   support

   Unrelated       0.95      0.97      0.96      1520
    Paralle

In [74]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
set_seed(2025)
# 数据导入
task1_data_path = r"D:\mystudy\2024新版本\Thirdpaperdata\task1_dataset.csv"
data = pd.read_csv(task1_data_path)
# 超前滞后检测器
def compute_lag_with_encoding(time_series_A, time_series_B, max_lag=20, n_hid=16):
    """
    检测超前滞后关系并进行时间编码。
    :param time_series_A: A 的时间序列（numpy array）
    :param time_series_B: B 的时间序列（numpy array）
    :param max_lag: 最大滞后步
    :param n_hid: 编码器的隐层维度
    :return: 时间编码器生成的特征向量 (numpy array)
    """
    series_A = (time_series_A - np.mean(time_series_A)) / np.std(time_series_A)
    series_B = (time_series_B - np.mean(time_series_B)) / np.std(time_series_B)

    # 计算互相关
    cross_correlation = np.correlate(series_B, series_A, mode='full')
    mid = len(cross_correlation) // 2
    relevant_range = cross_correlation[mid - max_lag: mid + max_lag + 1]

    # 找到最优滞后步
    optimal_lag = np.argmax(np.abs(relevant_range)) - max_lag

    # 时间编码器
    delta_t = torch.tensor([optimal_lag], dtype=torch.float32)
    dim_t = torch.arange(0, n_hid // 2, dtype=torch.float32)
    div_term = torch.exp((2 * dim_t) / n_hid * (-np.log(10000.0)))
    sin_term = torch.sin(delta_t.unsqueeze(-1) * div_term)
    cos_term = torch.cos(delta_t.unsqueeze(-1) * div_term)
    time_encoding = torch.cat([sin_term, cos_term], dim=-1).numpy()

    return time_encoding.flatten()

# 提取原始时间序列和超前滞后特征
X_A = np.array([np.array(eval(row["time_series_A"])) for _, row in data.iterrows()])
X_B = np.array([np.array(eval(row["time_series_B"])) for _, row in data.iterrows()])
y = data["relation_label"].values  # 标签列 (0: 无关, 1: 平行, 2: 滞后, 3: 超前)

# 提取超前滞后特征并拼接到时间序列
lag_features = np.array([compute_lag_with_encoding(X_A[i], X_B[i]) for i in range(len(X_A))])  # (样本数, 编码特征维度)
time_steps = X_A.shape[1]  # 时间步长

# 将 lag_features 扩展为与时间步长匹配的形状
lag_features_expanded = np.repeat(lag_features[:, np.newaxis, :], time_steps, axis=1)  # (样本数, 时间步长, 编码特征维度)

# 将 X_A 和 X_B 扩展为与 lag_features_expanded 匹配的形状
X_A_expanded = X_A[..., np.newaxis]  # (样本数, 时间步长, 1)
X_B_expanded = X_B[..., np.newaxis]  # (样本数, 时间步长, 1)

# 拼接 X_A, X_B 和 lag_features
X_combined = np.concatenate((X_A_expanded, X_B_expanded, lag_features_expanded), axis=2)  # (样本数, 时间步长, 特征维度)

# 数据集划分
X_train, X_test, y_train, y_test = train_test_split(X_combined, y, test_size=0.3, stratify=y)

# 转换为 PyTorch 张量
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.long)
y_test_tensor = torch.tensor(y_test, dtype=torch.long)

# 数据加载器
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

# 定义 CNN + GRU 模型
class CNN_GRU_Net(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim, num_layers=1, kernel_size=3, dropout=0.2):
        super(CNN_GRU_Net, self).__init__()
        self.conv1d = nn.Conv1d(in_channels=input_dim, out_channels=64, kernel_size=kernel_size, padding=1)
        self.relu = nn.ReLU()
        self.gru = nn.GRU(input_size=64, hidden_size=hidden_dim, num_layers=num_layers, batch_first=True, dropout=dropout)
        self.fc = nn.Linear(hidden_dim, output_dim)
    
    def forward(self, x):
        # 输入形状: (batch, seq_len, input_dim)
        x = x.permute(0, 2, 1)  # 转换为 (batch, input_dim, seq_len) 以适配 Conv1d
        x = self.conv1d(x)      # 经过 1D 卷积
        x = self.relu(x)
        x = x.permute(0, 2, 1)  # 转换回 (batch, seq_len, conv_features) 以适配 GRU
        _, h_n = self.gru(x)    # h_n: (num_layers, batch, hidden_dim)
        h_n = h_n[-1]           # 取最后一层的隐藏状态
        out = self.fc(h_n)      # 全连接层分类
        return out

# 初始化模型
input_dim = X_combined.shape[2]  # 每个时间步的特征维度（A 和 B + 编码特征）
hidden_dim = 128  # 隐藏层维度
output_dim = 4  # 分类数（无关、平行、滞后、超前）
num_layers = 1  # GRU 层数
kernel_size = 3  # 卷积核大小

model = CNN_GRU_Net(input_dim, hidden_dim, output_dim, num_layers=num_layers, kernel_size=kernel_size)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

# 损失函数和优化器
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# 训练和测试逻辑保持不变

# 训练模型
print("开始训练 CNN + GRU 模型（四分类）...")
train_model(model, train_loader, criterion, optimizer, device, epochs=30)
print("训练完成！")

# 测试
y_true, y_pred = evaluate_model(model, test_loader, device)

# 评估
print("\n评估结果：")
print(f"Accuracy: {accuracy_score(y_true, y_pred):.4f}")
print(f"Macro F1 Score: {f1_score(y_true, y_pred, average='macro'):.4f}")
print("\nClassification Report:")
print(classification_report(y_true, y_pred, target_names=[
    "Unrelated", "Parallel", "Lagging", "Leading"
]))


C:\Users\10340\AppData\Roaming\Python\Python311\site-packages\torch\nn\modules\rnn.py:71: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.2 and num_layers=1
  warnings.warn("dropout option adds dropout after all but last "


开始训练 CNN + GRU 模型（四分类）...
Epoch 1/30, Loss: 61.9377
Epoch 2/30, Loss: 17.5991
Epoch 3/30, Loss: 14.9949
Epoch 4/30, Loss: 12.2228
Epoch 5/30, Loss: 11.7071
Epoch 6/30, Loss: 10.0042
Epoch 7/30, Loss: 8.7679
Epoch 8/30, Loss: 8.1289
Epoch 9/30, Loss: 7.9265
Epoch 10/30, Loss: 7.7701
Epoch 11/30, Loss: 6.7133
Epoch 12/30, Loss: 6.0543
Epoch 13/30, Loss: 6.7798
Epoch 14/30, Loss: 5.4034
Epoch 15/30, Loss: 5.0051
Epoch 16/30, Loss: 4.1134
Epoch 17/30, Loss: 4.5812
Epoch 18/30, Loss: 3.3201
Epoch 19/30, Loss: 3.0679
Epoch 20/30, Loss: 2.0552
Epoch 21/30, Loss: 2.4499
Epoch 22/30, Loss: 3.9834
Epoch 23/30, Loss: 3.7828
Epoch 24/30, Loss: 2.0638
Epoch 25/30, Loss: 0.5213
Epoch 26/30, Loss: 0.2605
Epoch 27/30, Loss: 0.1326
Epoch 28/30, Loss: 0.0789
Epoch 29/30, Loss: 0.0509
Epoch 30/30, Loss: 0.0395
训练完成！

评估结果：
Accuracy: 0.9787
Macro F1 Score: 0.9788

Classification Report:
              precision    recall  f1-score   support

   Unrelated       0.96      0.97      0.96      1520
    Paralle

GRU+CNN 任务2

In [75]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score, f1_score
from torch.utils.data import DataLoader, TensorDataset
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
set_seed(2025)

# 数据导入
task2_data_path = r"D:\mystudy\2024新版本\Thirdpaperdata\task2_dataset.csv"
data = pd.read_csv(task2_data_path)

# 直接使用原始时间序列数据
X_A = np.array([np.array(eval(row["time_series_A"])) for _, row in data.iterrows()])
X_B = np.array([np.array(eval(row["time_series_B"])) for _, row in data.iterrows()])
y = data["delay_label"].values  # 标签列 (0: 无关, 1: 平行, 2-7: 滞后/超前)

# 将两个时间序列堆叠在一起，形状为 (样本数, 时间步长, 2)
X = np.stack((X_A, X_B), axis=2)

# 数据集划分
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, stratify=y)

# 转换为 PyTorch 张量
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.long)
y_test_tensor = torch.tensor(y_test, dtype=torch.long)

# 数据加载器
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

# 定义 CNN + GRU 模型
class CNN_GRU_Net(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim, num_layers=1, kernel_size=3, dropout=0.2):
        super(CNN_GRU_Net, self).__init__()
        self.conv1d = nn.Conv1d(in_channels=2, out_channels=64, kernel_size=kernel_size, padding=1)
        self.relu = nn.ReLU()
        self.gru = nn.GRU(input_size=64, hidden_size=hidden_dim, num_layers=num_layers, batch_first=True, dropout=dropout)
        self.fc = nn.Linear(hidden_dim, output_dim)
    
    def forward(self, x):
        # 输入形状: (batch, seq_len, input_dim)
        x = x.permute(0, 2, 1)  # 转换为 (batch, input_dim, seq_len) 以适配 Conv1d
        x = self.conv1d(x)      # 经过 1D 卷积
        x = self.relu(x)
        x = x.permute(0, 2, 1)  # 转换回 (batch, seq_len, conv_features) 以适配 GRU
        _, h_n = self.gru(x)    # h_n: (num_layers, batch, hidden_dim)
        h_n = h_n[-1]           # 取最后一层的隐藏状态
        out = self.fc(h_n)      # 全连接层分类
        return out

# 初始化模型
input_dim = 2  # 每个时间步的特征维度（A 和 B 两个时间序列）
hidden_dim = 128  # 隐藏层维度
output_dim = 8  # 分类数（无关、平行、滞后5/10/20、超前-5/-10/-20）
num_layers = 1  # GRU 层数
kernel_size = 3  # 卷积核大小

model = CNN_GRU_Net(input_dim, hidden_dim, output_dim, num_layers=num_layers, kernel_size=kernel_size)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

# 损失函数和优化器
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# 训练模型
def train_model(model, train_loader, criterion, optimizer, device, epochs=25):
    model.train()
    for epoch in range(epochs):
        total_loss = 0
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)

            # 前向传播
            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)

            # 反向传播和优化
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_loss += loss.item()
        print(f"Epoch {epoch+1}/{epochs}, Loss: {total_loss:.4f}")

# 测试模型
def evaluate_model(model, test_loader, device):
    model.eval()
    all_preds = []
    all_labels = []
    with torch.no_grad():
        for X_batch, y_batch in test_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            outputs = model(X_batch)
            _, preds = torch.max(outputs, 1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(y_batch.cpu().numpy())
    return np.array(all_labels), np.array(all_preds)

# 训练
print("开始训练 CNN + GRU 模型（八分类）...")
train_model(model, train_loader, criterion, optimizer, device, epochs=30)
print("训练完成！")

# 测试
y_true, y_pred = evaluate_model(model, test_loader, device)

# 评估
print("\n评估结果：")
print(f"Accuracy: {accuracy_score(y_true, y_pred):.4f}")
print(f"Macro F1 Score: {f1_score(y_true, y_pred, average='macro'):.4f}")
print("\nClassification Report:")
print(classification_report(y_true, y_pred, target_names=[
    "Unrelated", "Parallel", "Lagging-5", "Lagging-10", "Lagging-20",
    "Leading-5", "Leading-10", "Leading-20"
]))

C:\Users\10340\AppData\Roaming\Python\Python311\site-packages\torch\nn\modules\rnn.py:71: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.2 and num_layers=1
  warnings.warn("dropout option adds dropout after all but last "


开始训练 CNN + GRU 模型（八分类）...
Epoch 1/30, Loss: 179.3191
Epoch 2/30, Loss: 73.9235
Epoch 3/30, Loss: 53.6765
Epoch 4/30, Loss: 43.8889
Epoch 5/30, Loss: 36.4430
Epoch 6/30, Loss: 30.2493
Epoch 7/30, Loss: 28.7772
Epoch 8/30, Loss: 23.5708
Epoch 9/30, Loss: 22.0339
Epoch 10/30, Loss: 19.6602
Epoch 11/30, Loss: 17.8282
Epoch 12/30, Loss: 17.3987
Epoch 13/30, Loss: 13.4492
Epoch 14/30, Loss: 15.2894
Epoch 15/30, Loss: 10.4161
Epoch 16/30, Loss: 11.2209
Epoch 17/30, Loss: 7.9207
Epoch 18/30, Loss: 8.7666
Epoch 19/30, Loss: 9.9662
Epoch 20/30, Loss: 7.3898
Epoch 21/30, Loss: 8.0691
Epoch 22/30, Loss: 3.8087
Epoch 23/30, Loss: 5.6178
Epoch 24/30, Loss: 10.7588
Epoch 25/30, Loss: 6.4626
Epoch 26/30, Loss: 6.1723
Epoch 27/30, Loss: 6.4279
Epoch 28/30, Loss: 5.5078
Epoch 29/30, Loss: 4.1700
Epoch 30/30, Loss: 2.6330
训练完成！

评估结果：
Accuracy: 0.9490
Macro F1 Score: 0.9337

Classification Report:
              precision    recall  f1-score   support

   Unrelated       0.95      0.98      0.96      1500

In [78]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
set_seed(2025)
# 数据导入
task2_data_path = r"D:\mystudy\2024新版本\Thirdpaperdata\task2_dataset.csv"
data = pd.read_csv(task2_data_path)
# 超前滞后检测器
def compute_lag_with_encoding(time_series_A, time_series_B, max_lag=20, n_hid=16):
    """
    检测超前滞后关系并进行时间编码。
    :param time_series_A: A 的时间序列（numpy array）
    :param time_series_B: B 的时间序列（numpy array）
    :param max_lag: 最大滞后步
    :param n_hid: 编码器的隐层维度
    :return: 时间编码器生成的特征向量 (numpy array)
    """
    series_A = (time_series_A - np.mean(time_series_A)) / np.std(time_series_A)
    series_B = (time_series_B - np.mean(time_series_B)) / np.std(time_series_B)

    # 计算互相关
    cross_correlation = np.correlate(series_B, series_A, mode='full')
    mid = len(cross_correlation) // 2
    relevant_range = cross_correlation[mid - max_lag: mid + max_lag + 1]

    # 找到最优滞后步
    optimal_lag = np.argmax(np.abs(relevant_range)) - max_lag

    # 时间编码器
    delta_t = torch.tensor([optimal_lag], dtype=torch.float32)
    dim_t = torch.arange(0, n_hid // 2, dtype=torch.float32)
    div_term = torch.exp((2 * dim_t) / n_hid * (-np.log(10000.0)))
    sin_term = torch.sin(delta_t.unsqueeze(-1) * div_term)
    cos_term = torch.cos(delta_t.unsqueeze(-1) * div_term)
    time_encoding = torch.cat([sin_term, cos_term], dim=-1).numpy()

    return time_encoding.flatten()

# 提取原始时间序列和超前滞后特征
X_A = np.array([np.array(eval(row["time_series_A"])) for _, row in data.iterrows()])
X_B = np.array([np.array(eval(row["time_series_B"])) for _, row in data.iterrows()])
y = data["delay_label"].values  # 标签列 (0: 无关, 1: 平行, 2-7: 滞后/超前)

# 提取超前滞后特征并拼接到时间序列
lag_features = np.array([compute_lag_with_encoding(X_A[i], X_B[i]) for i in range(len(X_A))])  # (样本数, 编码特征维度)
time_steps = X_A.shape[1]  # 时间步长

# 将 lag_features 扩展为与时间步长匹配的形状
lag_features_expanded = np.repeat(lag_features[:, np.newaxis, :], time_steps, axis=1)  # (样本数, 时间步长, 编码特征维度)

# 将 X_A 和 X_B 扩展为与 lag_features_expanded 匹配的形状
X_A_expanded = X_A[..., np.newaxis]  # (样本数, 时间步长, 1)
X_B_expanded = X_B[..., np.newaxis]  # (样本数, 时间步长, 1)

# 拼接 X_A, X_B 和 lag_features
X_combined = np.concatenate((X_A_expanded, X_B_expanded, lag_features_expanded), axis=2)  # (样本数, 时间步长, 特征维度)

# 数据集划分
X_train, X_test, y_train, y_test = train_test_split(X_combined, y, test_size=0.3, stratify=y)

# 转换为 PyTorch 张量
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.long)
y_test_tensor = torch.tensor(y_test, dtype=torch.long)

# 数据加载器
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

# 定义 CNN + GRU 模型
class CNN_GRU_Net(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim, num_layers=1, kernel_size=3, dropout=0.2):
        super(CNN_GRU_Net, self).__init__()
        self.conv1d = nn.Conv1d(in_channels=input_dim, out_channels=64, kernel_size=kernel_size, padding=1)
        self.relu = nn.ReLU()
        self.gru = nn.GRU(input_size=64, hidden_size=hidden_dim, num_layers=num_layers, batch_first=True, dropout=dropout)
        self.fc = nn.Linear(hidden_dim, output_dim)
    
    def forward(self, x):
        # 输入形状: (batch, seq_len, input_dim)
        x = x.permute(0, 2, 1)  # 转换为 (batch, input_dim, seq_len) 以适配 Conv1d
        x = self.conv1d(x)      # 经过 1D 卷积
        x = self.relu(x)
        x = x.permute(0, 2, 1)  # 转换回 (batch, seq_len, conv_features) 以适配 GRU
        _, h_n = self.gru(x)    # h_n: (num_layers, batch, hidden_dim)
        h_n = h_n[-1]           # 取最后一层的隐藏状态
        out = self.fc(h_n)      # 全连接层分类
        return out

# 初始化模型
input_dim = X_combined.shape[2]  # 每个时间步的特征维度（A 和 B + 编码特征）
hidden_dim = 128  # 隐藏层维度
output_dim = 8  # 分类数（无关、平行、滞后5/10/20、超前-5/-10/-20）
num_layers = 1  # GRU 层数
kernel_size = 3  # 卷积核大小

model = CNN_GRU_Net(input_dim, hidden_dim, output_dim, num_layers=num_layers, kernel_size=kernel_size)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

# 损失函数和优化器
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# 训练和测试逻辑保持不变

# 训练模型
print("开始训练 CNN + GRU 模型（八分类）...")
train_model(model, train_loader, criterion, optimizer, device, epochs=30)
print("训练完成！")

# 测试
y_true, y_pred = evaluate_model(model, test_loader, device)

# 评估
print("\n评估结果：")
print(f"Accuracy: {accuracy_score(y_true, y_pred):.4f}")
print(f"Macro F1 Score: {f1_score(y_true, y_pred, average='macro'):.4f}")
print("\nClassification Report:")
print(classification_report(y_true, y_pred, target_names=[
    "Unrelated", "Parallel", "Lagging-5", "Lagging-10", "Lagging-20",
    "Leading-5", "Leading-10", "Leading-20"
]))


C:\Users\10340\AppData\Roaming\Python\Python311\site-packages\torch\nn\modules\rnn.py:71: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.2 and num_layers=1
  warnings.warn("dropout option adds dropout after all but last "


开始训练 CNN + GRU 模型（八分类）...
Epoch 1/30, Loss: 149.9377
Epoch 2/30, Loss: 51.6919
Epoch 3/30, Loss: 39.9197
Epoch 4/30, Loss: 33.6601
Epoch 5/30, Loss: 27.7174
Epoch 6/30, Loss: 25.6013
Epoch 7/30, Loss: 22.2716
Epoch 8/30, Loss: 21.4470
Epoch 9/30, Loss: 18.5770
Epoch 10/30, Loss: 19.5418
Epoch 11/30, Loss: 15.1447
Epoch 12/30, Loss: 15.3069
Epoch 13/30, Loss: 14.5449
Epoch 14/30, Loss: 14.1615
Epoch 15/30, Loss: 12.4708
Epoch 16/30, Loss: 10.6822
Epoch 17/30, Loss: 10.2384
Epoch 18/30, Loss: 10.1297
Epoch 19/30, Loss: 9.4487
Epoch 20/30, Loss: 8.2565
Epoch 21/30, Loss: 7.0402
Epoch 22/30, Loss: 6.5772
Epoch 23/30, Loss: 9.9679
Epoch 24/30, Loss: 5.9984
Epoch 25/30, Loss: 6.6908
Epoch 26/30, Loss: 5.5529
Epoch 27/30, Loss: 3.4914
Epoch 28/30, Loss: 3.9564
Epoch 29/30, Loss: 4.0626
Epoch 30/30, Loss: 5.7206
训练完成！

评估结果：
Accuracy: 0.9622
Macro F1 Score: 0.9538

Classification Report:
              precision    recall  f1-score   support

   Unrelated       0.96      0.97      0.96      150